In [268]:
import json
import pandas as pd
import numpy as np
import plotly.express as px
from tqdm.notebook import tqdm

In [275]:
data = []
for val in ['a', 'b', 'c']:
    with open(f'planes_09_{val}.json') as f:
        tdata = json.load(f)
        data.extend(tdata)
data = data[:8639]

In [278]:
data[0]['hourMin']

'Time: 00:00:1'

In [279]:
planes_dfs = []
for d in tqdm(data):
  temp = pd.DataFrame(d['planes'].values())
  v = d['hourMin'].replace('Time: ', '') + '0'
  date = "2025-01-09"
  temp['time'] = pd.to_datetime(date + ' ' + v)
  temp['latitude'] = temp['position'].apply(lambda x: x[1])
  temp['longitude'] = temp['position'].apply(lambda x: x[0])
  temp['altitude'] = temp['altitude'].apply(lambda x: 0 if x=='ground' else int(x))

#   temp['time'] = temp['time'].dt.tz_localize('UTC').dt.tz_convert('America/Los_Angeles')
  planes_dfs.append(temp)
planes_df = pd.concat(planes_dfs)

  0%|          | 0/8639 [00:00<?, ?it/s]

In [280]:
planes_df

,altitude,altitudeTime,category,country,dataSource,flight,flightTs,icao,icaoType,icaoTypeCache,...,position,position_time,prev_speed,prev_time,rotation,typeDescription,typeLong,time,latitude,longitude
0,525,1736380810,None,Spain,adsb,None,0,345109,A332,A332,...,"[-118.449321, 33.93045]",1736380810,132.1,1736380810,82.872832,L2J,AIRBUS A-330-200,2025-01-09 00:00:10,33.930450,-118.449321
1,0,1736380810,None,United Kingdom,adsb,VIR8Y,1736380810,407699,A35K,A35K,...,"[-118.425808, 33.946485]",1736380810,18.5,1736380810,262.687011,L2J,AIRBUS A-350-1000,2025-01-09 00:00:10,33.946485,-118.425808
2,10025,1736380810,None,United Kingdom,adsb,BAW8DS,1736380800,407995,B77W,B77W,...,"[-117.956842, 34.077713]",1736380810,257.6,1736380810,75.342998,L2J,BOEING 777-300ER,2025-01-09 00:00:10,34.077713,-117.956842
3,0,1736380800,None,United Arab Emirates,mlat,UAE37V,1736380800,896473,A388,A388,...,"[-118.412709, 33.947545]",1736380800,33.0,1736380800,0.000000,L4J,AIRBUS A-380-800,2025-01-09 00:00:10,33.947545,-118.412709
4,0,1736380810,None,United States,adsb,SKW6394,1736380810,a651e1,E75L,E75L,...,"[-118.413405, 33.93742]",1736380810,26.5,1736380810,262.816593,L2J,EMBRAER ERJ-170-200 (long wing),2025-01-09 00:00:10,33.937420,-118.413405
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,0,1736467180,None,United States,adsb,N50LK,1736467170,a6389b,GLF4,GLF4,...,"[-118.490024, 34.197297]",1736467180,0.0,1736467180,39.594391,L2J,GULFSTREAM 4,2025-01-09 23:59:50,34.197297,-118.490024
412,20000,1736467180,None,United States,adsb,SIS187,1736467180,a15a4f,C56X,C56X,...,"[-117.543983, 34.58524]",1736467180,402.8,1736467180,0.000000,L2J,CESSNA 560XL Citation XLS,2025-01-09 23:59:50,34.585240,-117.543983
413,11275,1736467190,None,United States,adsb,UAL1945,1736467180,a583f5,B739,B739,...,"[-117.55743, 33.985228]",1736467190,295.2,1736467190,281.837252,L2J,BOEING 737-900,2025-01-09 23:59:50,33.985228,-117.557430
414,0,1736467190,None,United States,adsb,OVJ54,1736467180,a0da26,GLF5,GLF5,...,"[-118.141694, 33.820713]",1736467190,0.0,1736467190,89.999997,L2J,GULFSTREAM 5,2025-01-09 23:59:50,33.820713,-118.141694


In [281]:
def filter_relevant(df):
    print("initial", len(df))
    lng1, lat1, lng2, lat2 = -119,34.0022,-117,34.4836
    
    flights = df[(df['latitude'] >= lat1) & (df['latitude'] <= lat2) & \
     (df['longitude'] >= lng1) & (df['longitude'] <= lng2)].icao.unique()
    print("flights", len(flights))
    
    bad_flights1 = df[(df['latitude'] >= lat1) & (df['latitude'] <= lat2) & \
     (df['longitude'] >= lng1) & (df['longitude'] <= lng2) & (df['altitude'] > 15000)].icao.unique()
    print("bad_flights1", len(bad_flights1))

    bad_flights2 = df.groupby('icao').size().reset_index(name='counts').query('counts < 100').icao.unique()
    print("bad_flights1", len(bad_flights2))


    bad_flights = np.concatenate([bad_flights1, bad_flights2])

    flights2 = [x for x in flights if (x not in bad_flights and '~' not in x)]
    print("good_flights", len(flights2))

    
    t = df[df['icao'].isin(flights2)]
    print("fin", len(t))


    aircraft_types = {
        'AERO Commander 680': 'Private Aircraft',
        'AEROSPATIALE AS-332 Super Puma': 'Helicopter',
        'AEROSPATIALE AS-350 Ecureuil': 'Helicopter',
        'AGUSTA AW-109 Grand': 'Helicopter',
        'AGUSTA AW-139': 'Helicopter',
        'AGUSTA AW-169': 'Helicopter',
        'AIRBUS A-300-600': 'Passenger Plane',
        'AIRBUS A-319': 'Passenger Plane',
        'AIRBUS A-320': 'Passenger Plane',
        'AIRBUS A-320neo': 'Passenger Plane',
        'AIRBUS A-321': 'Passenger Plane',
        'AIRBUS A-321neo': 'Passenger Plane',
        'AIRBUS A-330-200': 'Passenger Plane',
        'AIRBUS A-350-1000': 'Passenger Plane',
        'AIRBUS A-350-900': 'Passenger Plane',
        'AIRBUS A-380-800': 'Passenger Plane',
        'AIRBUS HELICOPTERS EC-135/635': 'Helicopter',
        'AIRPLANE FACTORY Sling 4': 'Private Aircraft',
        'AUTOGYRO Cavalon': 'Other',
        'AVRO RJ-85 Avroliner': 'Passenger Plane',
        'BEECH 200 Super King Air': 'Private Aircraft',
        'BEECH 300 Super King Air': 'Private Aircraft',
        'BEECH 33 Bonanza': 'Private Aircraft',
        'BEECH 35 Bonanza': 'Private Aircraft',
        'BEECH 36 Bonanza': 'Private Aircraft',
        'BEECH 36 Turbine Bonanza': 'Private Aircraft',
        'BEECH 55 Baron': 'Private Aircraft',
        'BEECH 58 Baron': 'Private Aircraft',
        'BEECH 58TC Baron': 'Private Aircraft',
        'BEECH 65 Queen Air': 'Private Aircraft',
        'BEECH 90 King Air': 'Private Aircraft',
        'BEECH 95 Travel Air': 'Private Aircraft',
        'BEECH 99 Airliner': 'Passenger Plane',
        'BEECH B36TC Bonanza': 'Private Aircraft',
        'BEECHCRAFT Super King Air 350': 'Private Aircraft',
        'BELL 206 JetRanger': 'Helicopter',
        'BELL 407': 'Helicopter',
        'BELL 412': 'Helicopter',
        'BELL 429 GlobalRanger': 'Helicopter',
        'BELL 505 JetRanger X': 'Helicopter',
        'BELL UH-1 Huey': 'Military Aircraft',
        'BELL-BOEING V-22 Osprey': 'Military Aircraft',
        'BOEING 737 MAX 8': 'Passenger Plane',
        'BOEING 737 MAX 9': 'Passenger Plane',
        'BOEING 737-700': 'Passenger Plane',
        'BOEING 737-800': 'Passenger Plane',
        'BOEING 737-900': 'Passenger Plane',
        'BOEING 747-8': 'Passenger Plane',
        'BOEING 757-200': 'Passenger Plane',
        'BOEING 757-300': 'Passenger Plane',
        'BOEING 767-300': 'Passenger Plane',
        'BOEING 767-400': 'Passenger Plane',
        'BOEING 777-200': 'Passenger Plane',
        'BOEING 777-200LR': 'Passenger Plane',
        'BOEING 777-300ER': 'Passenger Plane',
        'BOEING 787-10 Dreamliner': 'Passenger Plane',
        'BOEING 787-8 Dreamliner': 'Passenger Plane',
        'BOEING 787-9 Dreamliner': 'Passenger Plane',
        'BOEING-VERTOL CH-47 Chinook': 'Military Aircraft',
        'BOMBARDIER BD-100 Challenger 300': 'Private Jet',
        'BOMBARDIER BD-700 Global 5000/5500': 'Private Jet',
        'BOMBARDIER BD-700 Global 6000/6500': 'Private Jet',
        'BOMBARDIER CL-600 Challenger': 'Private Jet',
        'BOMBARDIER Regional Jet CRJ-700': 'Passenger Plane',
        'BRITISH AEROSPACE 146-200 Quiet Trader': 'Passenger Plane',
        'CANADAIR CL-415 SuperScooper': 'Firefighting Aircraft',
        'CANADAIR Cosmopolitan': 'Passenger Plane',
        'CESSNA 180 Skywagon': 'Private Aircraft',
        'CESSNA 150': 'Private Aircraft',
        'CESSNA 152': 'Private Aircraft',
        'CESSNA 170': 'Private Aircraft',
        'CESSNA 172 Skyhawk': 'Private Aircraft',
        'CESSNA 172R Cutlass RG': 'Private Aircraft',
        'CESSNA 175 Skylark': 'Private Aircraft',
        'CESSNA 182 Skylane': 'Private Aircraft',
        'CESSNA 201 Centurion': 'Private Aircraft',
        'CESSNA 208 Caravan': 'Cargo Aircraft',
        'CESSNA 400 Corvalis TT': 'Private Aircraft',
        'CESSNA 408 SkyCourier': 'Cargo Aircraft',
        'CESSNA 525A Citation CJ2': 'Private Jet',
        'CESSNA 525C Citation CJ4': 'Private Jet',
        'CESSNA 560 Citation Ultra': 'Private Jet',
        'CESSNA 560XL Citation XLS': 'Private Jet',
        'CESSNA 680 Citation Latitude': 'Private Jet',
        'CESSNA 750 Citation 10': 'Private Jet',
        'CESSNA R182 Skylane RG': 'Private Aircraft',
        'CESSNA T206 Turbo Stationair': 'Private Aircraft',
        'CESSNA T210 Turbo Centurion': 'Private Aircraft',
        'CHRISTEN Husky': 'Private Aircraft',
        'CIRRUS SF-50 Vision': 'Private Aircraft',
        'CIRRUS SR-20': 'Private Aircraft',
        'CIRRUS SR-22': 'Private Aircraft',
        'CIRRUS SR-22T Turbo': 'Private Aircraft',
        'DAHER Kodiak 100': 'Private Aircraft',
        'DASSAULT Falcon 2000': 'Private Jet',
        'DASSAULT Falcon 50': 'Private Jet',
        'DASSAULT Mirage F1': 'Military Aircraft',
        'DIAMOND DA-62': 'Private Aircraft',
        'EMBRAER EMB-120 Brasilia': 'Passenger Plane',
        'EMBRAER EMB-500 Phenom 100': 'Private Jet',
        'EMBRAER EMB-505 Phenom 300': 'Private Jet',
        'EMBRAER EMB-550 Praetor 600': 'Private Jet',
        'EMBRAER ERJ-135': 'Passenger Plane',
        'EMBRAER ERJ-145': 'Passenger Plane',
        'EMBRAER ERJ-170-200 (long wing)': 'Passenger Plane',
        'EMBRAER ERJ-190-400': 'Passenger Plane',
        'GRUMMAN S-2 Turbo Tracker': 'Firefighting Aircraft',
        'GULFSTREAM 2': 'Private Jet',
        'GULFSTREAM 4': 'Private Jet',
        'GULFSTREAM 5': 'Private Jet',
        'GULFSTREAM G650': 'Private Jet',
        'HAWKER BEECHCRAFT Hawker 750/850': 'Private Jet',
        'HONDA HondaJet': 'Private Jet',
        'HUGHES 500': 'Helicopter',
        'KAMAN K-Max': 'Helicopter',
        'LANCAIR Legacy': 'Private Aircraft',
        'LEARJET 45': 'Private Jet',
        'LEARJET 60': 'Private Jet',
        'LOCKHEED C-130 Hercules': 'Military Aircraft',
        'MCDONNELL DOUGLAS DC-10': 'Passenger Plane',
        'MCDONNELL-DOUGLAS MD-11': 'Passenger Plane',
        'MCDONNELL-DOUGLAS MD-520N': 'Helicopter',
        'MCDONNELL-DOUGLAS MD-87': 'Passenger Plane',
        'MOONEY M-20': 'Private Aircraft',
        'NORTH AMERICAN OV-10 Bronco': 'Military Aircraft',
        'NORTH AMERICAN ROCKWELL Turbo Commander 690/840': 'Private Aircraft',
        'PILATUS PC-12': 'Private Aircraft',
        'PIPER PA-20 Pacer': 'Private Aircraft',
        'PIPER PA-24 Comanche': 'Private Aircraft',
        'PIPER PA-28-140/150/160/180': 'Private Aircraft',
        'PIPER PA-28-201T/235/236': 'Private Aircraft',
        'PIPER PA-28R-180/200/201': 'Private Aircraft',
        'PIPER PA-31T2-620 Cheyenne 2': 'Private Aircraft',
        'PIPER PA-32': 'Private Aircraft',
        'PIPER PA-46-310/350': 'Private Aircraft',
        'RAYTHEON 390 Premier 1': 'Private Jet',
        'ROBINSON R-22 Mariner': 'Helicopter',
        'ROBINSON R-44 Raven': 'Helicopter',
        'ROBINSON R-66': 'Helicopter',
        'ROTORSPORT MTOSport': 'Other',
        'SIKORSKY S-61 Sea King': 'Military Aircraft',
        'SIKORSKY S-61R': 'Military Aircraft',
        'SIKORSKY S-64 Skycrane': 'Helicopter',
        'SIKORSKY S-76 Spirit': 'Helicopter',
        'SIKORSKY UH-60 Black Hawk': 'Military Aircraft',
        'SOCATA TBM-850': 'Private Aircraft',
        'SOCATA TBM-900/910/930/940': 'Private Aircraft',
        'SWEARINGEN Metro': 'Passenger Plane',
        'TECNAM P-2006T': 'Private Aircraft',
        'VANS RV-6': 'Private Aircraft',
        'VANS RV-7': 'Private Aircraft',
        'ZENAIR CH-2000 Zenith': 'Private Aircraft'
        }
    t = t[t['typeLong'].isna() == False]
    t['aircraft_type'] = t['typeLong'].apply(lambda x: aircraft_types[x] if x in aircraft_types else 'Other')

    t = t[~t['aircraft_type'].isin(['Passenger Plane', 'Private Jet', 'Private Aircraft'])]

    print("filtered", len(t))
    
    
    
    
    t = t[['icao', 'time', 'latitude', 'longitude', 'altitude', 'prev_speed',
           'rotation', 'icaoType', 'typeDescription', 'typeLong', 'military', 'dataSource',
           'aircraft_type']]
    
    print(t.groupby('icao').size().reset_index(name='counts'))

    return t


In [282]:
relevant_flights = filter_relevant(planes_df)

initial 2197331
flights 2423
bad_flights1 1019
bad_flights1 1362
good_flights 599
fin 546002
filtered 162848
       icao  counts
0    a02d2e     701
1    a03088     431
2    a0427c     228
3    a0c89b    1184
4    a0c8c7     401
..      ...     ...
127  adc5a7     674
128  ae5ca0     288
129  ae6b94    1265
130  c06f07    1031
131  c06f09    1359

[132 rows x 2 columns]


In [283]:
relevant_flights.sort_values(['icao', 'time'], inplace=True)

In [284]:

# fig = px.line(relevant_flights, x='longitude', y='latitude', color='icao', 
#         line_group='icao', hover_name='icao', markers=True)

# fig.update_traces(mode='markers+lines', opacity=0.5)


In [285]:
import plotly.express as px

def plot_on_map(df):
    # Create the scatter_geo plot
    fig = px.scatter_geo(
        df,
        lat='latitude',
        lon='longitude',
        color='icao',
        hover_name='icao',
        title="Flight Paths on a Geographic Map",
        width=1200,
        height=800
    )

    # Add lines connecting the points for each 'icao'
    fig.update_traces(
        mode='markers+lines',
        marker=dict(size=1),  # Adjust marker size if needed
        line=dict(width=0.2)  # Adjust line width and opacity
    )

    # Set the geographic projection
    fig.update_geos(
        projection_type="mercator",  # Choose the desired map projection
        showland=True,
        landcolor="lightgray",
        showocean=True,
        oceancolor="lightblue",
        showcoastlines=True,
        coastlinecolor="gray",
        lataxis_range=[33.5, 34.5], 
        lonaxis_range=[-119, -117]
    )

    fig.show()


In [286]:
relevant_flights.to_csv('flights_v2.csv', index=False)

In [287]:
# plot_on_map(relevant_flights)

In [288]:
plot_on_map(relevant_flights[relevant_flights['icao'] == 'ad4fa3'])

In [289]:
relevant_flights.head().T

,498,498,486,486,486
icao,a02d2e,a02d2e,a02d2e,a02d2e,a02d2e
time,2025-01-09 20:07:10,2025-01-09 20:07:20,2025-01-09 20:07:30,2025-01-09 20:07:40,2025-01-09 20:07:50
latitude,34.259747,34.261963,34.264495,34.268417,34.271339
longitude,-118.407555,-118.409636,-118.411674,-118.410981,-118.408298
altitude,925,1000,1100,1200,1300
prev_speed,29.8,65.2,77.8,89.0,91.9
rotation,349.196988,322.184357,326.368117,8.307768,37.190505
icaoType,B412,B412,B412,B412,B412
typeDescription,H2T,H2T,H2T,H2T,H2T
typeLong,BELL 412,BELL 412,BELL 412,BELL 412,BELL 412


In [290]:
# group by these keys and create list of others
# icao, icaoType, typeDescription, typeLong, aircraft_type
# lat lng should be array of coordinates (lng, lat)
# the time should be array of timestamps
oof = relevant_flights.groupby([
  'icao',
  'icaoType',
  'typeDescription',
    'typeLong',
    'aircraft_type'
]).agg({
    'latitude': lambda x: list(x),
    'longitude': lambda x: list(x),
    'altitude': lambda x: list(x),
    'time': lambda x: list(x.astype(int)//1e9),
    'dataSource': 'count'
    }).reset_index()

oof['coordinates'] = oof.apply(lambda x: list(zip(x['longitude'], x['latitude'])), axis=1)

oof.drop(['latitude', 'longitude'], axis=1, inplace=True)
oof.to_json('flights_v3.json', orient='records')


In [291]:
oof.sort_values('dataSource', ascending=False)
# zz = np.array(oof[oof['icao'] == 'acd27a']['coordinates'].values[0])
# px.line(x=zz[:,0], y=zz[:,1])


,icao,icaoType,typeDescription,typeLong,aircraft_type,altitude,time,dataSource,coordinates
116,acd27a,AS50,H1T,AEROSPATIALE AS-350 Ecureuil,Helicopter,"[7300, 7350, 7325, 7275, 7300, 7300, 7300, 725...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",6992,"[(-118.521629, 34.124268), (-118.514398, 34.12..."
23,a32af8,A139,H2T,AGUSTA AW-139,Helicopter,"[1775, 1700, 1700, 1725, 1775, 1800, 1850, 182...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",6833,"[(-118.49699, 34.120044), (-118.49596, 34.1254..."
21,a3238a,A139,H2T,AGUSTA AW-139,Helicopter,"[950, 1050, 1200, 1300, 1425, 1525, 1575, 1625...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",6159,"[(-118.510208, 34.080102), (-118.509464, 34.08..."
31,a40442,AS50,H1T,AEROSPATIALE AS-350 Ecureuil,Helicopter,"[2925, 2950, 2975, 2925, 2875, 2900, 2900, 290...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",5936,"[(-118.238983, 34.236517), (-118.239291, 34.23..."
24,a32eaf,A139,H2T,AGUSTA AW-139,Helicopter,"[0, 0, 1050, 1150, 1175, 1300, 1425, 1500, 152...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",5184,"[(-118.4864, 34.126785), (-118.4864, 34.126785..."
...,...,...,...,...,...,...,...,...,...
82,a8d9e2,H500,H1T,HUGHES 500,Helicopter,"[2025, 2100, 2200, 2225, 2225, 2325, 2450, 250...","[1736457430.0, 1736457440.0, 1736457450.0, 173...",163,"[(-118.213921, 34.740026), (-118.211231, 34.73..."
35,a4c786,UH1,H1T,BELL UH-1 Huey,Military Aircraft,"[6575, 6525, 6525, 6450, 6450, 6400, 6400, 640...","[1736446560.0, 1736446570.0, 1736446580.0, 173...",151,"[(-118.536259, 34.914551), (-118.542144, 34.90..."
76,a7d631,MRF1,L1J,DASSAULT Mirage F1,Military Aircraft,"[-175, -75, 675, 1250, 2175, 3300, 4550, 5375,...","[1736463880.0, 1736463890.0, 1736463900.0, 173...",148,"[(-119.127839, 34.114243), (-119.120384, 34.12..."
30,a3c231,C25B,L2J,CESSNA 525B Citation CJ3,Other,"[11850, 11700, 11550, 11250, 11100, 10950, 108...","[1736440530.0, 1736440540.0, 1736440550.0, 173...",131,"[(-117.548103, 33.957856), (-117.56052, 33.962..."


In [292]:
oof[oof['icao'] == 'acd27a'].time.values[0][-1]

1736467190.0

In [295]:
oof.groupby(['aircraft_type']).count()

,icao,icaoType,typeDescription,typeLong,altitude,time,dataSource,coordinates
aircraft_type,,,,,,,,
Cargo Aircraft,13,13,13,13,13,13,13,13
Firefighting Aircraft,7,7,7,7,7,7,7,7
Helicopter,72,72,72,72,72,72,72,72
Military Aircraft,36,36,36,36,36,36,36,36
Other,4,4,4,4,4,4,4,4


In [296]:
oof

,icao,icaoType,typeDescription,typeLong,aircraft_type,altitude,time,dataSource,coordinates
0,a02d2e,B412,H2T,BELL 412,Helicopter,"[925, 1000, 1100, 1200, 1300, 1300, 1400, 1450...","[1736453230.0, 1736453240.0, 1736453250.0, 173...",701,"[(-118.407555, 34.259747), (-118.409636, 34.26..."
1,a03088,B407,H1T,BELL 407,Helicopter,"[325, 375, 450, 525, 525, 625, 650, 775, 800, ...","[1736448940.0, 1736448950.0, 1736448960.0, 173...",431,"[(-118.931351, 34.389395), (-118.927787, 34.38..."
2,a0427c,H60,H2T,SIKORSKY UH-60 Black Hawk,Military Aircraft,"[6925, 6925, 6925, 6800, 6725, 6625, 6525, 632...","[1736458180.0, 1736458190.0, 1736458200.0, 173...",228,"[(-117.54407, 34.741653), (-117.553879, 34.738..."
3,a0c89b,UH1,H1T,BELL UH-1 Huey,Military Aircraft,"[2200, 2200, 2200, 2200, 2200, 2200, 2200, 220...","[1736413600.0, 1736413610.0, 1736413620.0, 173...",1184,"[(-118.211714, 34.739136), (-118.211714, 34.73..."
4,a0c8c7,R66,H1T,ROBINSON R-66,Helicopter,"[175, 250, 350, 425, 425, 475, 475, 475, 500, ...","[1736381690.0, 1736381700.0, 1736381710.0, 173...",401,"[(-118.247452, 33.889424), (-118.249998, 33.89..."
...,...,...,...,...,...,...,...,...,...
127,adc5a7,C208,L1T,CESSNA 208 Caravan,Cargo Aircraft,"[8825, 8825, 8825, 8825, 8825, 8825, 8825, 882...","[1736385500.0, 1736385510.0, 1736385520.0, 173...",674,"[(-118.454418, 34.473887), (-118.447864, 34.46..."
128,ae5ca0,H60,H2T,SIKORSKY UH-60 Black Hawk,Military Aircraft,"[2700, 2700, 2800, 2800, 2800, 2800, 2800, 280...","[1736461500.0, 1736461510.0, 1736461520.0, 173...",288,"[(-119.403877, 34.383436), (-119.393806, 34.37..."
129,ae6b94,V22,R2T,BELL-BOEING V-22 Osprey,Military Aircraft,"[8325, 8325, 8325, 8325, 8325, 8325, 8325, 832...","[1736447960.0, 1736447970.0, 1736447980.0, 173...",1265,"[(-117.543658, 33.269806), (-117.557074, 33.28..."
130,c06f07,CL2T,A2T,CANADAIR CL-415 SuperScooper,Firefighting Aircraft,"[3900, 3900, 3900, 3900, 3900, 3900, 3900, 380...","[1736380840.0, 1736380850.0, 1736380860.0, 173...",1031,"[(-118.42252, 34.431082), (-118.42252, 34.4310..."


In [297]:
oof.sort_values('dataSource', ascending=False)

,icao,icaoType,typeDescription,typeLong,aircraft_type,altitude,time,dataSource,coordinates
116,acd27a,AS50,H1T,AEROSPATIALE AS-350 Ecureuil,Helicopter,"[7300, 7350, 7325, 7275, 7300, 7300, 7300, 725...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",6992,"[(-118.521629, 34.124268), (-118.514398, 34.12..."
23,a32af8,A139,H2T,AGUSTA AW-139,Helicopter,"[1775, 1700, 1700, 1725, 1775, 1800, 1850, 182...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",6833,"[(-118.49699, 34.120044), (-118.49596, 34.1254..."
21,a3238a,A139,H2T,AGUSTA AW-139,Helicopter,"[950, 1050, 1200, 1300, 1425, 1525, 1575, 1625...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",6159,"[(-118.510208, 34.080102), (-118.509464, 34.08..."
31,a40442,AS50,H1T,AEROSPATIALE AS-350 Ecureuil,Helicopter,"[2925, 2950, 2975, 2925, 2875, 2900, 2900, 290...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",5936,"[(-118.238983, 34.236517), (-118.239291, 34.23..."
24,a32eaf,A139,H2T,AGUSTA AW-139,Helicopter,"[0, 0, 1050, 1150, 1175, 1300, 1425, 1500, 152...","[1736380810.0, 1736380820.0, 1736380830.0, 173...",5184,"[(-118.4864, 34.126785), (-118.4864, 34.126785..."
...,...,...,...,...,...,...,...,...,...
82,a8d9e2,H500,H1T,HUGHES 500,Helicopter,"[2025, 2100, 2200, 2225, 2225, 2325, 2450, 250...","[1736457430.0, 1736457440.0, 1736457450.0, 173...",163,"[(-118.213921, 34.740026), (-118.211231, 34.73..."
35,a4c786,UH1,H1T,BELL UH-1 Huey,Military Aircraft,"[6575, 6525, 6525, 6450, 6450, 6400, 6400, 640...","[1736446560.0, 1736446570.0, 1736446580.0, 173...",151,"[(-118.536259, 34.914551), (-118.542144, 34.90..."
76,a7d631,MRF1,L1J,DASSAULT Mirage F1,Military Aircraft,"[-175, -75, 675, 1250, 2175, 3300, 4550, 5375,...","[1736463880.0, 1736463890.0, 1736463900.0, 173...",148,"[(-119.127839, 34.114243), (-119.120384, 34.12..."
30,a3c231,C25B,L2J,CESSNA 525B Citation CJ3,Other,"[11850, 11700, 11550, 11250, 11100, 10950, 108...","[1736440530.0, 1736440540.0, 1736440550.0, 173...",131,"[(-117.548103, 33.957856), (-117.56052, 33.962..."
